In [ ]:
## Morges 2024 Forecast 

In [ ]:
import xarray as xr
forecast_Morges = xr.open_dataset("/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc")
forecast_Morges 

<xarray.Dataset> Size: 187MB
Dimensions:                  (forecast_reference_time: 1, lead_time: 34,
                              realization: 11, y: 292, x: 427)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * lead_time                (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 ...
  * realization              (realization) float64 88B 0.0 1.0 2.0 ... 9.0 10.0
  * y                        (y) float64 2kB 1.042e+06 1.042e+06 ... 1.332e+06
  * x                        (x) float64 3kB 2.44e+06 2.442e+06 ... 2.866e+06
    time                     (forecast_reference_time, lead_time) datetime64[ns] 272B ...
Data variables:
    precipitation_amount     (forecast_reference_time, lead_time, realization, y, x) float32 187MB ...
Attributes:
    source:        ICON-CH1-EPS
    history:       Produced by fieldextra version v15.0.2 (v15.0.2) on 2025-0...
    grid_mapping:  {"epsg_code": "EPSG:2056",\n    "long_name": "Swiss coordi...
    crs:           EPSG:2056

In [17]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
from pathlib import Path
import rasterio
import geopandas as gpd
from shapely.geometry import box

import requests
from PIL import Image
from io import BytesIO

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"
catchment_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/geo_ezgg_40km.gpkg"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/MORGES_FORECAST_PLOTS/")
out_dir.mkdir(exist_ok=True, parents=True)

# plot extent in EPSG:2056
plot_extent = (2510227.993, 2540975.818, 1140564.666, 1172820.076)   # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Swisstopo WMS layer
wms_layer = "ch.swisstopo.swisstlm3d-karte-grau"

precip_alpha = 0.70
wms_start_resolution_m = 2
wms_max_pixels = 40_000_000
save_dpi = 400

# choose lead times
max_lead_hours = 6

# optionally force variable name
forced_var_name = None
# example:
# forced_var_name = "precipitation_amount"

# --------------------------------------------------
# Robust WMS fetch
# --------------------------------------------------
def get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=2,
    layer="ch.swisstopo.swisstlm3d-karte-grau",
    max_pixels=10_000_000
):
    xmin, xmax = float(min(xmin, xmax)), float(max(xmin, xmax))
    ymin, ymax = float(min(ymin, ymax)), float(max(ymin, ymax))
    dx, dy = xmax - xmin, ymax - ymin

    res = float(resolution_m)
    while True:
        width_px = int(np.ceil(dx / res))
        height_px = int(np.ceil(dy / res))
        if width_px * height_px <= max_pixels:
            break
        res *= 2

    bbox = f"{xmin},{ymin},{xmax},{ymax}"
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer,
        "BBOX": bbox,
        "CRS": "EPSG:2056",
        "WIDTH": width_px,
        "HEIGHT": height_px,
        "FORMAT": "image/png",
        "TRANSPARENT": "TRUE",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/png,image/*,*/*;q=0.8"
    }

    r = requests.get("https://wms.geo.admin.ch/", params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        print("Failed to fetch WMS:", r.status_code)
        return None, res

    ctype = r.headers.get("Content-Type", "")
    if "image" not in ctype.lower():
        print("WMS returned non-image content:", ctype)
        print(r.text[:250])
        return None, res

    return Image.open(BytesIO(r.content)).convert("RGBA"), res


# --------------------------------------------------
# LOAD FORECAST
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)

print(ds)
print("Data variables:", list(ds.data_vars))

if forced_var_name is not None:
    var_name = forced_var_name
else:
    var_name = list(ds.data_vars)[0]

da = ds[var_name]
print(f"Using variable: {var_name}")
print("Original dims:", da.dims)

# remove forecast_reference_time if present
if "forecast_reference_time" in da.dims:
    da = da.isel(forecast_reference_time=0)

# reorder
required_dims = {"lead_time", "realization", "y", "x"}
missing = required_dims - set(da.dims)
if missing:
    raise ValueError(f"Missing expected dimensions: {missing}")

da = da.transpose("lead_time", "realization", "y", "x")
print("Reordered dims:", da.dims)

# --------------------------------------------------
# SELECT FIRST 6 LEAD HOURS
# --------------------------------------------------
lead_time_hours = da["lead_time"] / np.timedelta64(1, "h")
lead_mask = (lead_time_hours > 0) & (lead_time_hours <= max_lead_hours)
da = da.sel(lead_time=da["lead_time"][lead_mask])

selected_hours = (da["lead_time"] / np.timedelta64(1, "h")).values
print("Selected lead times (h):", selected_hours)

# --------------------------------------------------
# READ DEM BOUNDS
# --------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# --------------------------------------------------
# LOAD CATCHMENTS
# --------------------------------------------------
catchment_gdf = gpd.read_file(catchment_file)

if catchment_gdf.crs is None:
    raise ValueError("Catchment file has no CRS.")

if catchment_gdf.crs.to_epsg() != 2056:
    catchment_gdf = catchment_gdf.to_crs("EPSG:2056")

bbox_geom = box(xmin, ymin, xmax, ymax)

try:
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:2056")
    catchment_plot = gpd.clip(catchment_gdf, bbox_gdf)
except Exception as e:
    print("gpd.clip failed, using intersects fallback.")
    print("Reason:", e)
    catchment_plot = catchment_gdf[catchment_gdf.intersects(bbox_geom)]

print("Catchment features loaded:", len(catchment_gdf))
print("Catchment features in plot window:", len(catchment_plot))

# --------------------------------------------------
# SPATIAL SUBSET
# --------------------------------------------------
x_ascending = bool(da.x.values[0] < da.x.values[-1])
y_ascending = bool(da.y.values[0] < da.y.values[-1])

x_slice = slice(xmin, xmax) if x_ascending else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if y_ascending else slice(ymax, ymin)

da_space = da.sel(x=x_slice, y=y_slice)

print("Subset shape after spatial selection:", da_space.shape)

if da_space.sizes["x"] == 0 or da_space.sizes["y"] == 0:
    raise ValueError("Spatial subset is empty. Check plot_extent vs forecast coordinates.")

# extent from selected coordinates
xvals = da_space.x.values
yvals = da_space.y.values
data_extent = (float(xvals.min()), float(xvals.max()), float(yvals.min()), float(yvals.max()))

# --------------------------------------------------
# COLOR LEVELS
# --------------------------------------------------
levels_full = np.array(
    [0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350],
    dtype=float
)

vmax = float(np.nanmax(da_space.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in selected forecast subset.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if len(levels) < 2:
    levels = np.array([0.1, max(1.0, vmax)])

if levels[-1] < vmax:
    higher_levels = levels_full[levels_full > levels[-1]]
    if len(higher_levels) > 0:
        levels = np.append(levels, higher_levels[0])
    else:
        levels = np.append(levels, vmax)

mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"Forecast vmax in window: {vmax:.2f}")
print("Levels used:", levels)

# --------------------------------------------------
# GET BACKGROUND ONCE
# --------------------------------------------------
bg_img, used_res = get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=wms_start_resolution_m,
    layer=wms_layer,
    max_pixels=wms_max_pixels
)
print(f"WMS resolution used: {used_res} m/px")

# --------------------------------------------------
# MORGES LOCATION
# --------------------------------------------------
morges_x = 2527132.921
morges_y = 1150707.111

# --------------------------------------------------
# PLOT PER ENSEMBLE AND LEAD TIME
# --------------------------------------------------
lead_hours = (da_space["lead_time"] / np.timedelta64(1, "h")).values
realizations = da_space["realization"].values

for ens in realizations:
    for lt, lt_h in zip(da_space["lead_time"].values, lead_hours):

        frame = da_space.sel(realization=ens, lead_time=lt).astype(float)
        frame = frame.where(frame > 0)

        fig, ax = plt.subplots(figsize=(8, 7))
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")

        # Background
        if bg_img is not None:
            ax.imshow(bg_img, extent=(xmin, xmax, ymin, ymax), origin="upper", zorder=0)

        # Forecast precipitation
        im = ax.imshow(
            frame.values,
            extent=data_extent,
            origin="lower",
            interpolation="nearest",
            cmap=cmap,
            norm=norm,
            alpha=precip_alpha,
            zorder=2
        )

        # Catchment outlines
        if len(catchment_plot) > 0:
            catchment_plot.boundary.plot(
                ax=ax,
                edgecolor="black",
                linewidth=1.2,
                zorder=4
            )

        # DEM rectangle
        rect = Rectangle(
            (dem_left, dem_bottom),
            dem_right - dem_left,
            dem_top - dem_bottom,
            fill=False,
            edgecolor="red",
            linewidth=2.0,
            zorder=5
        )
        ax.add_patch(rect)

        # Morges marker
        ax.scatter(
            morges_x,
            morges_y,
            marker="^",
            s=50,
            edgecolor="black",
            facecolor="yellow",
            linewidth=1.5,
            zorder=6
        )

        ax.text(
            morges_x + 500,
            morges_y + 500,
            "Morges",
            fontsize=8,
            color="black",
            weight="bold",
            zorder=6
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect("equal")

        title_txt = f"Ensemble {int(ens)} | Lead time +{float(lt_h):.0f} h"
        ax.set_title(title_txt, color="black")

        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
        cbar.set_label("precipitation (mm)")
        cbar.ax.tick_params(colors="black")
        cbar.outline.set_edgecolor("black")

        out = out_dir / f"ICON_MORGES_ens{int(ens):02d}_lead{int(round(float(lt_h))):02d}h.png"
        plt.savefig(out, dpi=save_dpi, transparent=True, bbox_inches="tight")
        plt.close()

print("Done. Saved to:", out_dir)

<xarray.Dataset> Size: 187MB
Dimensions:                  (forecast_reference_time: 1, lead_time: 34,
                              realization: 11, y: 292, x: 427)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * lead_time                (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 ...
  * realization              (realization) float64 88B 0.0 1.0 2.0 ... 9.0 10.0
  * y                        (y) float64 2kB 1.042e+06 1.042e+06 ... 1.332e+06
  * x                        (x) float64 3kB 2.44e+06 2.442e+06 ... 2.866e+06
    time                     (forecast_reference_time, lead_time) datetime64[ns] 272B ...
Data variables:
    precipitation_amount     (forecast_reference_time, lead_time, realization, y, x) float32 187MB ...
Attributes:
    source:        ICON-CH1-EPS
    history:       Produced by fieldextra version v15.0.2 (v15.0.2) on 2025-0...
    grid_mapping:  {"epsg_code": "EPSG:2056",\n    "long_name": "Swiss coordi...
    crs:

In [9]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
forecast_path = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"

window_hours = 3

# ---------------------------------------------------
# LOAD FORECAST
# ---------------------------------------------------
ds = xr.open_dataset(forecast_path)

print("Dataset:")
print(ds)
print("Data variables:", list(ds.data_vars))

da = ds["precipitation_amount"]
print(f"Using variable: {da.name}")
print("Original dims:", da.dims)
print("Attrs:", da.attrs)

if "forecast_reference_time" in da.dims:
    da = da.isel(forecast_reference_time=0)

required_dims = {"lead_time", "realization", "y", "x"}
missing = required_dims - set(da.dims)
if missing:
    raise ValueError(f"Missing expected dimensions: {missing}")

da = da.transpose("lead_time", "realization", "y", "x")
print("Reordered dims:", da.dims)

# ---------------------------------------------------
# SELECT +1h TO +3h
# ---------------------------------------------------
lead_time_hours = da["lead_time"] / np.timedelta64(1, "h")
mask_3h = (lead_time_hours > 0) & (lead_time_hours <= window_hours)
da_3h = da.sel(lead_time=da["lead_time"][mask_3h])

print("Selected lead times (hours):", (da_3h["lead_time"] / np.timedelta64(1, "h")).values)

# ---------------------------------------------------
# ACCUMULATE OVER 3 HOURS
# ---------------------------------------------------
da_3h_acc = da_3h.sum(dim="lead_time")

print("Accumulated 3h global min:", float(da_3h_acc.min()))
print("Accumulated 3h global max:", float(da_3h_acc.max()))

# ---------------------------------------------------
# READ DEM BOUNDS
# ---------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# ---------------------------------------------------
# SPATIAL SUBSET TO DEM DOMAIN
# ---------------------------------------------------
x_ascending = bool(da_3h_acc.x.values[0] < da_3h_acc.x.values[-1])
y_ascending = bool(da_3h_acc.y.values[0] < da_3h_acc.y.values[-1])

x_slice = slice(dem_left, dem_right) if x_ascending else slice(dem_right, dem_left)
y_slice = slice(dem_bottom, dem_top) if y_ascending else slice(dem_top, dem_bottom)

da_dem = da_3h_acc.sel(x=x_slice, y=y_slice)

print("Subset shape over DEM domain:", da_dem.shape)

if da_dem.sizes["x"] == 0 or da_dem.sizes["y"] == 0:
    raise ValueError("DEM spatial subset is empty. Check DEM bounds vs forecast coordinates.")

# ---------------------------------------------------
# METRICS PER ENSEMBLE OVER DEM DOMAIN
# ---------------------------------------------------
rows = []

for ens in da_dem["realization"].values:
    arr = da_dem.sel(realization=ens).values
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        domain_sum_3h = np.nan
        domain_mean_3h = np.nan
        domain_p80_3h = np.nan
        domain_p90_3h = np.nan
        domain_p99_9_3h = np.nan
        domain_max_3h = np.nan
        n_pixels = 0
    else:
        domain_sum_3h = np.sum(arr)
        domain_mean_3h = np.mean(arr)
        domain_p80_3h = np.percentile(arr, 80)
        domain_p90_3h = np.percentile(arr, 90)
        domain_p99_9_3h = np.percentile(arr, 99.9)
        domain_max_3h = np.max(arr)
        n_pixels = len(arr)

    rows.append({
        "realization": int(ens),
        "n_pixels_dem_domain": n_pixels,
        "domain_sum_3h": domain_sum_3h,
        "domain_mean_3h": domain_mean_3h,
        "domain_p80_3h": domain_p80_3h,
        "domain_p90_3h": domain_p90_3h,
        "domain_p99_9_3h": domain_p99_9_3h,
        "domain_max_3h": domain_max_3h
    })

df_dem = pd.DataFrame(rows)

pd.set_option("display.float_format", "{:.10f}".format)

print("\nMetrics over DEM domain:")
print(df_dem)

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------
out_csv = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/morges_dem_domain_sum_mean_p80_p90_p99_9_max_3h_per_ensemble.csv"
df_dem.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

Dataset:
<xarray.Dataset> Size: 187MB
Dimensions:                  (forecast_reference_time: 1, lead_time: 34,
                              realization: 11, y: 292, x: 427)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * lead_time                (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 ...
  * realization              (realization) float64 88B 0.0 1.0 2.0 ... 9.0 10.0
  * y                        (y) float64 2kB 1.042e+06 1.042e+06 ... 1.332e+06
  * x                        (x) float64 3kB 2.44e+06 2.442e+06 ... 2.866e+06
    time                     (forecast_reference_time, lead_time) datetime64[ns] 272B ...
Data variables:
    precipitation_amount     (forecast_reference_time, lead_time, realization, y, x) float32 187MB ...
Attributes:
    source:        ICON-CH1-EPS
    history:       Produced by fieldextra version v15.0.2 (v15.0.2) on 2025-0...
    grid_mapping:  {"epsg_code": "EPSG:2056",\n    "long_name": "Swiss coordi...

In [ ]:
#### THE SAME AS ABOVETHE CODE HOWEVER in here I am adding the levels 1, levels 2, levels 3 and calcualted it based o nthe percentile 80 

In [11]:
## Adding the levels 
import numpy as np
import pandas as pd
import xarray as xr
import rasterio

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
forecast_path = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"

window_hours = 3

# thresholds
thr_L1 = 15
thr_L2 = 25
thr_L3 = 35

# ---------------------------------------------------
# LOAD FORECAST
# ---------------------------------------------------
ds = xr.open_dataset(forecast_path)

print("Dataset:")
print(ds)
print("Data variables:", list(ds.data_vars))

da = ds["precipitation_amount"]
print(f"Using variable: {da.name}")
print("Original dims:", da.dims)
print("Attrs:", da.attrs)

if "forecast_reference_time" in da.dims:
    da = da.isel(forecast_reference_time=0)

required_dims = {"lead_time", "realization", "y", "x"}
missing = required_dims - set(da.dims)
if missing:
    raise ValueError(f"Missing expected dimensions: {missing}")

da = da.transpose("lead_time", "realization", "y", "x")
print("Reordered dims:", da.dims)

# ---------------------------------------------------
# SELECT +1h TO +3h
# ---------------------------------------------------
lead_time_hours = da["lead_time"] / np.timedelta64(1, "h")
mask_3h = (lead_time_hours > 0) & (lead_time_hours <= window_hours)
da_3h = da.sel(lead_time=da["lead_time"][mask_3h])

print("Selected lead times (hours):", (da_3h["lead_time"] / np.timedelta64(1, "h")).values)

# ---------------------------------------------------
# ACCUMULATE OVER 3 HOURS
# ---------------------------------------------------
da_3h_acc = da_3h.sum(dim="lead_time")

print("Accumulated 3h global min:", float(da_3h_acc.min()))
print("Accumulated 3h global max:", float(da_3h_acc.max()))

# ---------------------------------------------------
# READ DEM BOUNDS
# ---------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# ---------------------------------------------------
# SPATIAL SUBSET TO DEM DOMAIN
# ---------------------------------------------------
x_ascending = bool(da_3h_acc.x.values[0] < da_3h_acc.x.values[-1])
y_ascending = bool(da_3h_acc.y.values[0] < da_3h_acc.y.values[-1])

x_slice = slice(dem_left, dem_right) if x_ascending else slice(dem_right, dem_left)
y_slice = slice(dem_bottom, dem_top) if y_ascending else slice(dem_top, dem_bottom)

da_dem = da_3h_acc.sel(x=x_slice, y=y_slice)

print("Subset shape over DEM domain:", da_dem.shape)

if da_dem.sizes["x"] == 0 or da_dem.sizes["y"] == 0:
    raise ValueError("DEM spatial subset is empty. Check DEM bounds vs forecast coordinates.")

# ---------------------------------------------------
# FUNCTION TO ASSIGN LEVEL
# ---------------------------------------------------
def classify_level(value, thr_L1=15, thr_L2=25, thr_L3=35):
    if np.isnan(value):
        return "NoData"
    elif value < thr_L1:
        return "Below_L1"
    elif value < thr_L2:
        return "L1"
    elif value < thr_L3:
        return "L2"
    else:
        return "L3"

# ---------------------------------------------------
# METRICS PER ENSEMBLE OVER DEM DOMAIN
# ---------------------------------------------------
rows = []

for ens in da_dem["realization"].values:
    arr = da_dem.sel(realization=ens).values
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        domain_sum_3h = np.nan
        domain_mean_3h = np.nan
        domain_p80_3h = np.nan
        domain_p90_3h = np.nan
        domain_p99_9_3h = np.nan
        domain_max_3h = np.nan
        n_pixels = 0
    else:
        domain_sum_3h = np.sum(arr)
        domain_mean_3h = np.mean(arr)
        domain_p80_3h = np.percentile(arr, 80)
        domain_p90_3h = np.percentile(arr, 90)
        domain_p99_9_3h = np.percentile(arr, 99.9)
        domain_max_3h = np.max(arr)
        n_pixels = len(arr)

    # choose metric for threshold logic
    trigger_metric = domain_p80_3h
    level = classify_level(trigger_metric, thr_L1, thr_L2, thr_L3)

    rows.append({
        "realization": int(ens),
        "n_pixels_dem_domain": n_pixels,
        "domain_sum_3h": domain_sum_3h,
        "domain_mean_3h": domain_mean_3h,
        "domain_p80_3h": domain_p80_3h,
        "domain_p90_3h": domain_p90_3h,
        "domain_p99_9_3h": domain_p99_9_3h,
        "domain_max_3h": domain_max_3h,
        "trigger_metric_used": trigger_metric,
        "level": level
    })

df_dem = pd.DataFrame(rows)

pd.set_option("display.float_format", "{:.10f}".format)

print("\nMetrics over DEM domain:")
print(df_dem)

# ---------------------------------------------------
# COUNT ENSEMBLES PER LEVEL
# ---------------------------------------------------
df_counts = (
    df_dem["level"]
    .value_counts(dropna=False)
    .rename_axis("level")
    .reset_index(name="n_ensembles")
)

print("\nNumber of ensembles in each level:")
print(df_counts)

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------
out_csv = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/morges_dem_domain_with_levels_3h_per_ensemble.csv"
df_dem.to_csv(out_csv, index=False)

out_counts_csv = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/morges_dem_domain_level_counts_3h.csv"
df_counts.to_csv(out_counts_csv, index=False)

print("\nSaved:")
print(out_csv)
print(out_counts_csv)

Dataset:
<xarray.Dataset> Size: 187MB
Dimensions:                  (forecast_reference_time: 1, lead_time: 34,
                              realization: 11, y: 292, x: 427)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * lead_time                (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 ...
  * realization              (realization) float64 88B 0.0 1.0 2.0 ... 9.0 10.0
  * y                        (y) float64 2kB 1.042e+06 1.042e+06 ... 1.332e+06
  * x                        (x) float64 3kB 2.44e+06 2.442e+06 ... 2.866e+06
    time                     (forecast_reference_time, lead_time) datetime64[ns] 272B ...
Data variables:
    precipitation_amount     (forecast_reference_time, lead_time, realization, y, x) float32 187MB ...
Attributes:
    source:        ICON-CH1-EPS
    history:       Produced by fieldextra version v15.0.2 (v15.0.2) on 2025-0...
    grid_mapping:  {"epsg_code": "EPSG:2056",\n    "long_name": "Swiss coordi...

In [10]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
forecast_path = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"

window_hours = 6

# ---------------------------------------------------
# LOAD FORECAST
# ---------------------------------------------------
ds = xr.open_dataset(forecast_path)

print("Dataset:")
print(ds)
print("Data variables:", list(ds.data_vars))

da = ds["precipitation_amount"]
print(f"Using variable: {da.name}")
print("Original dims:", da.dims)
print("Attrs:", da.attrs)

if "forecast_reference_time" in da.dims:
    da = da.isel(forecast_reference_time=0)

required_dims = {"lead_time", "realization", "y", "x"}
missing = required_dims - set(da.dims)
if missing:
    raise ValueError(f"Missing expected dimensions: {missing}")

da = da.transpose("lead_time", "realization", "y", "x")
print("Reordered dims:", da.dims)

# ---------------------------------------------------
# SELECT +1h TO +6h
# ---------------------------------------------------
lead_time_hours = da["lead_time"] / np.timedelta64(1, "h")
lead_mask = (lead_time_hours > 0) & (lead_time_hours <= window_hours)
da_window = da.sel(lead_time=da["lead_time"][lead_mask])

print("Selected lead times (hours):", (da_window["lead_time"] / np.timedelta64(1, "h")).values)

# ---------------------------------------------------
# ACCUMULATE OVER 6 HOURS
# ---------------------------------------------------
da_window_acc = da_window.sum(dim="lead_time")

print(f"Accumulated {window_hours}h global min:", float(da_window_acc.min()))
print(f"Accumulated {window_hours}h global max:", float(da_window_acc.max()))

# ---------------------------------------------------
# READ DEM BOUNDS
# ---------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# ---------------------------------------------------
# SPATIAL SUBSET TO DEM DOMAIN
# ---------------------------------------------------
x_ascending = bool(da_window_acc.x.values[0] < da_window_acc.x.values[-1])
y_ascending = bool(da_window_acc.y.values[0] < da_window_acc.y.values[-1])

x_slice = slice(dem_left, dem_right) if x_ascending else slice(dem_right, dem_left)
y_slice = slice(dem_bottom, dem_top) if y_ascending else slice(dem_top, dem_bottom)

da_dem = da_window_acc.sel(x=x_slice, y=y_slice)

print("Subset shape over DEM domain:", da_dem.shape)

if da_dem.sizes["x"] == 0 or da_dem.sizes["y"] == 0:
    raise ValueError("DEM spatial subset is empty. Check DEM bounds vs forecast coordinates.")

# ---------------------------------------------------
# METRICS PER ENSEMBLE OVER DEM DOMAIN
# ---------------------------------------------------
rows = []

for ens in da_dem["realization"].values:
    arr = da_dem.sel(realization=ens).values
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        domain_sum = np.nan
        domain_mean = np.nan
        domain_p80 = np.nan
        domain_p90 = np.nan
        domain_p99_9 = np.nan
        domain_max = np.nan
        n_pixels = 0
    else:
        domain_sum = np.sum(arr)
        domain_mean = np.mean(arr)
        domain_p80 = np.percentile(arr, 80)
        domain_p90 = np.percentile(arr, 90)
        domain_p99_9 = np.percentile(arr, 99.9)
        domain_max = np.max(arr)
        n_pixels = len(arr)

    rows.append({
        "realization": int(ens),
        "n_pixels_dem_domain": n_pixels,
        f"domain_sum_{window_hours}h": domain_sum,
        f"domain_mean_{window_hours}h": domain_mean,
        f"domain_p80_{window_hours}h": domain_p80,
        f"domain_p90_{window_hours}h": domain_p90,
        f"domain_p99_9_{window_hours}h": domain_p99_9,
        f"domain_max_{window_hours}h": domain_max
    })

df_dem = pd.DataFrame(rows)

pd.set_option("display.float_format", "{:.10f}".format)

print(f"\nMetrics over DEM domain for {window_hours} hours:")
print(df_dem)

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------
out_csv = f"/storage/homefs/ge24z347/Zell_event/Data_forprocess/morges_dem_domain_sum_mean_p80_p90_p99_9_max_{window_hours}h_per_ensemble.csv"
df_dem.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

Dataset:
<xarray.Dataset> Size: 187MB
Dimensions:                  (forecast_reference_time: 1, lead_time: 34,
                              realization: 11, y: 292, x: 427)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * lead_time                (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 ...
  * realization              (realization) float64 88B 0.0 1.0 2.0 ... 9.0 10.0
  * y                        (y) float64 2kB 1.042e+06 1.042e+06 ... 1.332e+06
  * x                        (x) float64 3kB 2.44e+06 2.442e+06 ... 2.866e+06
    time                     (forecast_reference_time, lead_time) datetime64[ns] 272B ...
Data variables:
    precipitation_amount     (forecast_reference_time, lead_time, realization, y, x) float32 187MB ...
Attributes:
    source:        ICON-CH1-EPS
    history:       Produced by fieldextra version v15.0.2 (v15.0.2) on 2025-0...
    grid_mapping:  {"epsg_code": "EPSG:2056",\n    "long_name": "Swiss coordi...